# Demo 04c — Naive Bayes for Regression

**Class 04 · Block 3** (slides: `slides/04_probabilistic_models.md`)

Naive Bayes is usually taught for classification, but the idea is general: model how the features are produced by the
target, assume independence given the target, invert with Bayes' rule. When the target $y$ is a **number** there are two
routes, and each works with **discrete** or **continuous** features. On the house-price data:

* **R1 — Discretize the target** into bins, classify the bin with Gaussian NB, predict the expected value.
* **R2 — Discrete features:** the same, with categorical (binned) features.
* **R3 — A continuous target:** $x_j \mid y$ is linear-Gaussian; the posterior of $y$ is a Gaussian in closed form.
* **R4 — Where the naive assumption hurts:** correlated features counted twice.

In [ ]:
import warnings
from pathlib import Path

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
import polars as pl
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.naive_bayes import CategoricalNB, GaussianNB
from sklearn.preprocessing import KBinsDiscretizer, StandardScaler

jax.config.update("jax_enable_x64", True)
plt.rcParams["figure.figsize"] = (9, 4)
DATA = Path("../../data")
warnings.filterwarnings("ignore", message="Bins whose width")      # bins of constant features collapse

In [ ]:
housing = pl.read_parquet(DATA / "california_housing.parquet")
for c in ("AveRooms", "AveBedrms", "Population", "AveOccup"):
    housing = housing.with_columns(pl.col(c).log())
names = [c for c in housing.columns if c != "MedHouseVal"]
X = housing.select(names).to_numpy().astype(float)
y = housing["MedHouseVal"].to_numpy().astype(float)
perm = np.random.default_rng(1).permutation(len(y))
te, tr = perm[: len(y) // 5], perm[len(y) // 5 :]
scaler = StandardScaler().fit(X[tr])
Xtr, Xte = np.asarray(scaler.transform(X[tr])), np.asarray(scaler.transform(X[te]))
ytr, yte = y[tr], y[te]


def report(name, pred):
    print(f"{name:<44} R2 {r2_score(yte, pred):.3f}  RMSE {mean_squared_error(yte, pred) ** 0.5:.3f}  MAE {mean_absolute_error(yte, pred):.3f}")


report("mean baseline", np.full(len(yte), ytr.mean()))
report("linear regression (Class 03)", LinearRegression().fit(Xtr, ytr).predict(Xte))

## R1 — Regression by classification: discretize the target

Cut $y$ into $K$ **bins** (quantile bins: the same number of districts in each). Each bin becomes a class $k$ with a
representative value $m_k$ (the mean of the training targets in the bin). A Naive Bayes classifier gives
$P(k \mid x)$, and the prediction is

$$\hat y(x) = \sum_{k=1}^{K} P(k \mid x)\; m_k \qquad\text{(the expected value)}$$

We could also return the *most probable* bin's $m_k$; the expectation uses the whole posterior.

In [ ]:
def target_bins(K):
    kb = KBinsDiscretizer(K, encode="ordinal", strategy="quantile").fit(ytr[:, None])
    cls = np.asarray(kb.transform(ytr[:, None])).ravel().astype(int)
    means = np.array([ytr[cls == k].mean() for k in range(K)])
    return cls, means


for K in (5, 10, 20):
    cls, means = target_bins(K)
    nb = GaussianNB().fit(Xtr, cls)
    P = nb.predict_proba(Xte)
    report(f"Gaussian NB, K = {K:2d}: expected value", P @ means[nb.classes_])
    report(f"Gaussian NB, K = {K:2d}: most probable bin", means[nb.classes_][P.argmax(axis=1)])

Using the **whole posterior** (the expectation) is clearly better than picking the most probable bin: the argmax
throws away everything but one number per bin. A bonus of the classification route: for every district we get a
*distribution* over prices, not only a prediction.

In [ ]:
cls, means = target_bins(10)
nb = GaussianNB().fit(Xtr, cls)
P = nb.predict_proba(Xte)
pred = P @ means[nb.classes_]

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
axes[0].scatter(yte, pred, s=3, alpha=0.3)
axes[0].plot([0, 5], [0, 5], "k--")
axes[0].set(xlabel="actual value", ylabel="predicted value (expectation)", title=f"K = 10: R2 = {r2_score(yte, pred):.3f}")
for i, color in zip((0, 1), ("tab:blue", "tab:red")):
    axes[1].bar(means[nb.classes_] + 0.05 * i, P[i], width=0.1, color=color, label=f"district {i}: y = {yte[i]:.2f}, E[y|x] = {pred[i]:.2f}")
axes[1].set(xlabel="bin value m_k", ylabel="P(bin | x)", title="the posterior over prices of two districts")
axes[1].legend(fontsize=7)
plt.tight_layout()
plt.show()

## R2 — Discrete features

The features can be made discrete as well: 5 quantile bins per feature, and `CategoricalNB` estimates
$P(\text{bin of } x_j \mid k)$ by counting (with Laplace smoothing), exactly like the word counts of the spam filter.

In [ ]:
fb = KBinsDiscretizer(5, encode="ordinal", strategy="quantile").fit(Xtr)
Ftr, Fte = np.asarray(fb.transform(Xtr)).astype(int), np.asarray(fb.transform(Xte)).astype(int)
for K in (5, 10, 20):
    cls, means = target_bins(K)
    cat = CategoricalNB(alpha=1.0, min_categories=5).fit(Ftr, cls)
    report(f"Categorical NB (5 bins/feature), K = {K:2d}", cat.predict_proba(Fte) @ means[cat.classes_])
for bins in (3, 5, 10, 20):
    fb_ = KBinsDiscretizer(bins, encode="ordinal", strategy="quantile").fit(Xtr)
    cls, means = target_bins(10)
    cat = CategoricalNB(alpha=1.0, min_categories=bins).fit(np.asarray(fb_.transform(Xtr)).astype(int), cls)
    report(f"Categorical NB, K = 10, {bins:2d} bins per feature", cat.predict_proba(np.asarray(fb_.transform(Xte)).astype(int)) @ means[cat.classes_])

The number of bins is the hyper-parameter, and it matters a lot: 3 bins lose most of the information ($R^2 = 0.42$),
5 bins 0.56, 10 bins 0.63, and **20 bins 0.66**, above the Gaussian model (0.57) and close to linear regression (0.68).
Counting in finer bins is a *non-parametric* estimate of each $P(x_j \mid k)$: it captures skewed and multi-modal
shapes that a Gaussian cannot. With more bins each cell holds fewer examples, so at some point the smoothing takes over.

## R3 — A continuous target: linear-Gaussian Naive Bayes

Keep $y$ continuous. Model each feature as **linear in the target plus noise**, independent given $y$, with a Gaussian
prior on $y$:

$$y \sim \mathcal{N}(\mu_y, \tau^2), \qquad x_j \mid y \sim \mathcal{N}(a_j + b_j\,y,\; s_j^2)$$

The parameters are fitted by maximum likelihood: $\mu_y$ and $\tau^2$ are the mean and variance of the targets;
$(a_j, b_j)$ is the least-squares fit of feature $j$ **on** $y$ (regressing $x$ on $y$, the opposite direction to
ordinary regression) and $s_j^2$ the residual variance. Bayes' rule gives the posterior of $y$ given $x$: a **Gaussian**,

$$P = \frac{1}{\tau^2} + \sum_j \frac{b_j^2}{s_j^2}, \qquad
\hat y = \mathbb{E}[y \mid x] = \frac{1}{P}\Big(\frac{\mu_y}{\tau^2} + \sum_j \frac{b_j\,(x_j - a_j)}{s_j^2}\Big),
\qquad \text{sd}(y\mid x) = \frac{1}{\sqrt P}$$

Each feature contributes **precision** $b_j^2/s_j^2$ (how informative it is about $y$) and a weighted vote.

In [ ]:
def fit_lgnb(Xa, ya):
    mu, tau2 = ya.mean(), ya.var()
    A = np.c_[ya, np.ones(len(ya))]
    coef = np.linalg.lstsq(A, Xa, rcond=None)[0]                     # x_j = b_j y + a_j
    b, a = coef[0], coef[1]
    s2 = ((Xa - A @ coef) ** 2).mean(axis=0)
    return mu, tau2, a, b, s2


def predict_lgnb(model, Xa):
    mu, tau2, a, b, s2 = model
    precision = 1 / tau2 + np.sum(b**2 / s2)
    mean = (mu / tau2 + ((Xa - a) * b / s2).sum(axis=1)) / precision
    return mean, np.sqrt(1 / precision)


model = fit_lgnb(Xtr, ytr)
mean, sd = predict_lgnb(model, Xte)
report("linear-Gaussian NB (continuous target)", mean)
print(pl.DataFrame({"feature": names, "a_j": model[2].round(3), "b_j (x per unit of y)": model[3].round(3), "s_j^2": model[4].round(3), "precision b^2/s^2": (model[3] ** 2 / model[4]).round(3)}))
print(f"posterior sd of y (the same for every district): {sd:.3f}   residual sd on the test set: {np.std(yte - mean):.3f}")

**Check with JAX.** The posterior mean is the maximum of the log-posterior. We can find it without the formula by
letting `jax.grad` climb the log-posterior of one district, and compare.

In [ ]:
mu, tau2, a, b, s2 = (jnp.asarray(v) for v in model)


def neg_log_posterior(y_, x):
    return 0.5 * (y_ - mu) ** 2 / tau2 + 0.5 * jnp.sum((x - a - b * y_) ** 2 / s2)


g = jax.jit(jax.grad(neg_log_posterior))
x0 = jnp.asarray(Xte[0])
y_ = jnp.asarray(0.0)
for _ in range(500):
    y_ = y_ - 0.1 * g(y_, x0)
print(f"district 0: closed form {mean[0]:.6f}   jax.grad ascent {float(y_):.6f}")

**Uncertainty.** The posterior is a Gaussian, so a 90% interval is $\hat y \pm 1.645\,\text{sd}$. Does it contain the
truth 90% of the time?

In [ ]:
inside = np.abs(yte - mean) < 1.645 * sd
print(f"90% interval half-width {1.645 * sd:.2f}: covers {inside.mean():.3f} of the test districts")
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
axes[0].scatter(yte, mean, s=3, alpha=0.3)
axes[0].plot([0, 5], [0, 5], "k--")
axes[0].set(xlabel="actual value", ylabel="posterior mean", title=f"linear-Gaussian NB: R2 = {r2_score(yte, mean):.3f}")
axes[1].hist(yte - mean, bins=50, density=True, alpha=0.6)
grid = np.linspace(-3, 3, 200)
axes[1].plot(grid, np.exp(-0.5 * (grid / sd) ** 2) / (sd * np.sqrt(2 * np.pi)), "r", label="model: N(0, sd^2)")
axes[1].set(xlabel="actual - predicted", title="the model's error distribution against the truth")
axes[1].legend()
plt.tight_layout()
plt.show()

The intervals are all the same width: this model has the same posterior variance for every district (a property of
its linear-Gaussian structure: the variance does not depend on $x$). Linear regression has the same limitation.
Discretizing the target (R1) gave a *different* posterior for each district.

## R4 — Where the naive assumption hurts

Naive Bayes treats the features as independent given $y$. If two features carry the same evidence, it counts it twice.
We append 0, 1, 3, 10 exact copies of the most informative feature (median income) and refit.

In [ ]:
j = names.index("MedInc")
print("copies of MedInc | R2 (linear regression) | R2 (linear-Gaussian NB) | posterior sd | 90% coverage")
for copies in (0, 1, 3, 10):
    Xa_tr = np.c_[Xtr, np.repeat(Xtr[:, [j]], copies, axis=1)] if copies else Xtr
    Xa_te = np.c_[Xte, np.repeat(Xte[:, [j]], copies, axis=1)] if copies else Xte
    m_lr = LinearRegression().fit(Xa_tr, ytr)
    m_nb = fit_lgnb(Xa_tr, ytr)
    mean_, sd_ = predict_lgnb(m_nb, Xa_te)
    print(f"{copies:16d} | {r2_score(yte, m_lr.predict(Xa_te)):23.3f} | {r2_score(yte, mean_):23.3f} | {sd_:12.3f} | {np.mean(np.abs(yte - mean_) < 1.645 * sd_):.3f}")

Linear regression is unaffected (the copies share one weight). Naive Bayes counts the income evidence again
for every copy: the posterior becomes **narrower** and more confident while the prediction gets worse, so the intervals
stop covering what they should. The same happens with correlated features that are not exact copies (the number of
rooms, bedrooms and occupancy).

## Comparison

| Model | Target | Features | Posterior |
|:--|:--|:--|:--|
| Gaussian NB on binned target | discretized | continuous | a distribution over $K$ bins, per district |
| Categorical NB on binned target | discretized | discrete (bins) | a distribution over $K$ bins, per district |
| linear-Gaussian NB | continuous | continuous | a Gaussian; the same width for every district |
| linear regression (Class 03) | continuous | continuous | no distribution (only a point) |

## Summary

* Naive Bayes needs $P(x_j \mid y)$; with a numeric $y$ either **discretize $y$** (and reuse the classifier) or
  model $x_j \mid y$ as **linear-Gaussian** and get a closed-form Gaussian posterior.
* Use the expectation of the posterior, not its mode.
* Independence makes it fast and gives a full posterior, but it is a *strong* assumption: it loses to linear regression
  when the features are correlated (as on this data), and it becomes over-confident when evidence is repeated.